In [ ]:
!pip install git+https://github.com/begelb/latent_dynamics.git@paper

# Section 5.3 - Chafee-Infante PDE with bistability

## What this notebook shows

The **Chafee-Infante** semilinear parabolic PDE
$u_t = u_{xx} + \lambda(u - u^3)$ on $[0, \pi]$ with Dirichlet boundary
conditions (paper section 5.3). At $\lambda = 28$ the equation has eleven steady
states -- two stable, nine unstable. Discretizing in space yields a
**64-dimensional** system; we learn a two-dimensional latent model whose Morse
graph recovers the **bistability** (the two stable steady states $u_\pm$) as two
minimal nodes.

### How to run

Edit the **parameters** cell below, then *Run All*. Three modes:

| `MODE` | what it does | cost |
|--------|--------------|------|
| `"replay"` | re-render the paper's saved Morse graph and Morse sets | seconds |
| `"morse"` | recompute the Morse graph of the *saved* model at your `SUBDIV` | seconds-minutes |
| `"retrain"` | run the whole pipeline from scratch with your `OVERRIDES` | minutes-hours |

**Toy subdivisions are a qualitative preview.** Coarse CMGDB grids can merge
nearby recurrent sets and change the Morse graph; the paper figures use the
config's (finer) values. The paper value for this example is noted in the
parameters cell.

> **On `retrain` for this example:** fresh retrains currently *overfit* and can
> fail the two-attractor ground truth, so the **replay artifacts are the paper
> reference**. Retrain mode is provided for experimentation, not verification.
> The config sets `compute_roa: true`; exact regions of attraction are slower
> under PyPI `cmgdb` than under the old fork.

In [ ]:
# ===== PARAMETERS  (edit, then Run All) ====================================
MODE = "replay"            # "replay" | "morse" | "retrain"
SEED = None                # None -> the config's default seed
SUBDIV = (10, 14, 20)      # MODE="morse": (subdiv_init, subdiv_min, subdiv_max)
                           # paper value: (10, 14, 28)
OVERRIDES = {}             # MODE="retrain": config overrides (pydantic-validated), e.g.
                           #   {"training": {"epochs": 300}, "cmgdb": {"subdiv_max": 20}}
BOX_SCALE = "auto"         # Morse-set box size: "auto" | float | {label: float}
# ===========================================================================

In [ ]:
from latentdynamics.replay import load_experiment, retrain

REPLAY_CONFIG  = "chafee_infante_replay"
RETRAIN_CONFIG = "chafee_infante"

if MODE == "replay":
    exp = load_experiment(REPLAY_CONFIG, seed=SEED)
elif MODE == "morse":
    exp = load_experiment(REPLAY_CONFIG, seed=SEED).recompute_morse(subdiv=SUBDIV)
elif MODE == "retrain":
    exp = retrain(RETRAIN_CONFIG, seed=SEED, overrides=OVERRIDES)
else:
    raise ValueError(f"unknown MODE {MODE!r}")
exp

## Morse graph

Two minimal nodes -- the two stable steady states.

In [ ]:
exp.show_morse_graph()

## Morse sets

In [ ]:
exp.show_morse_sets(box_scale=BOX_SCALE)

## Regions of attraction (optional)

When the run computed exact regions of attraction, the basin figure is saved
next to the Morse sets. Replay artifacts include it; toy `morse` recomputes do
not (RoA is left off there for speed).

In [ ]:
from latentdynamics.replay import show_image

roa_png = exp.morse_dir / "regions_of_attraction_exact.png"
if roa_png.exists():
    show_image(roa_png, width=520)
else:
    print("no exact-RoA figure for this run (expected for toy 'morse' mode)")

## Run provenance

In [ ]:
exp.diagnostics()